# Agent with Tools

> **Reading time:** ~10 min | **Goal:** Agent that can call functions and take actions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

In [ ]:
import sys; sys.path.insert(0, '../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
apply_course_theme()
apply_plot_theme()

In [ ]:
# Quick Win: Agent that can do math and get weather!

import anthropic
import json

client = anthropic.Anthropic()

# Define tools the agent can use
tools = [
    {
        "name": "calculator",
        "description": "Perform mathematical calculations",
        "input_schema": {
            "type": "object",
            "properties": {
                "expression": {"type": "string", "description": "Math expression like '2 + 2'"}
            },
            "required": ["expression"]
        }
    },
    {
        "name": "get_weather",
        "description": "Get current weather for a city",
        "input_schema": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "City name"}
            },
            "required": ["city"]
        }
    }
]

# Implement the tools
def run_tool(name: str, args: dict) -> str:
    if name == "calculator":
        return str(eval(args["expression"]))  # Simple eval for demo
    elif name == "get_weather":
        return f"72°F and sunny in {args['city']}"  # Mock response

# Agent loop
def ask_agent(question: str) -> str:
    messages = [{"role": "user", "content": question}]
    
    while True:
        response = client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=1024,
            tools=tools,
            messages=messages
        )
        
        # Check if done
        if response.stop_reason == "end_turn":
            return response.content[0].text
        
        # Handle tool calls
        for block in response.content:
            if block.type == "tool_use":
                result = run_tool(block.name, block.input)
                messages.append({"role": "assistant", "content": response.content})
                messages.append({"role": "user", "content": [
                    {"type": "tool_result", "tool_use_id": block.id, "content": result}
                ]})
                break

# Try it!
print(ask_agent("What's 1234 * 5678?"))
print(ask_agent("What's the weather in Tokyo?"))

## How It Works

```
User: "What's 2+2?"            Agent: "I'll calculate that"
        │                              │
        ▼                              ▼
┌──────────────┐              ┌──────────────┐
│     LLM      │─────────────▶│  Tool Call   │
│              │  tool_use    │ calculator   │
└──────────────┘              │ expr: "2+2"  │
        ▲                     └──────┬───────┘
        │                            │
        │     tool_result: "4"       ▼
        └────────────────────┌──────────────┐
                             │  Run Tool    │
                             │  return "4"  │
                             └──────────────┘
```

Loop: LLM decides → calls tool → gets result → continues until done

In [ ]:
# Modify This: Add your own tool!

# TODO: Add a tool that searches your database, calls an API, etc.
my_tools = tools + [
    {
        "name": "search_database",  # Change this
        "description": "Search company database for information",  # Change this
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "Search query"}
            },
            "required": ["query"]
        }
    }
]

def run_my_tool(name: str, args: dict) -> str:
    if name == "search_database":
        # TODO: Implement your actual search logic
        return f"Found results for: {args['query']}"
    return run_tool(name, args)  # Fallback to existing tools

## Copy-Paste Template

Production-ready tool-using agent:

In [ ]:
# Copy this entire cell for your projects

import anthropic
from typing import Callable

class ToolAgent:
    def __init__(self, system_prompt: str = "You are a helpful assistant with tools."):
        self.client = anthropic.Anthropic()
        self.system = system_prompt
        self.tools = []
        self.tool_handlers = {}
    
    def add_tool(self, name: str, description: str, parameters: dict, handler: Callable):
        """Register a tool the agent can use."""
        self.tools.append({
            "name": name,
            "description": description,
            "input_schema": {"type": "object", "properties": parameters, "required": list(parameters.keys())}
        })
        self.tool_handlers[name] = handler
    
    def run(self, user_input: str, max_turns: int = 10) -> str:
        """Run the agent until it produces a final response."""
        messages = [{"role": "user", "content": user_input}]
        
        for _ in range(max_turns):
            response = self.client.messages.create(
                model="claude-sonnet-4-20250514",
                max_tokens=1024,
                system=self.system,
                tools=self.tools,
                messages=messages
            )
            
            if response.stop_reason == "end_turn":
                for block in response.content:
                    if hasattr(block, 'text'):
                        return block.text
                return ""
            
            # Process tool calls
            messages.append({"role": "assistant", "content": response.content})
            tool_results = []
            
            for block in response.content:
                if block.type == "tool_use":
                    result = self.tool_handlers[block.name](**block.input)
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": str(result)
                    })
            
            messages.append({"role": "user", "content": tool_results})
        
        return "Max turns reached"

# Example usage
agent = ToolAgent()
agent.add_tool(
    name="multiply",
    description="Multiply two numbers",
    parameters={"a": {"type": "number"}, "b": {"type": "number"}},
    handler=lambda a, b: a * b
)
print(agent.run("What is 123 times 456?"))

## What's Next?

- **Multi-Agent Systems:** [../concepts/visual_guides/multi_agent.md](../concepts/visual_guides/multi_agent.md)
- **Production Template:** [../templates/agent_template.py](../templates/agent_template.py)
- **Build a Project:** [../projects/project_1_beginner/](../projects/project_1_beginner/)